# Transformer-Block

Jetzt haben wir Attention. Aber Attention allein reicht nicht aus.
Wir müssen sie in einen vollständigen Baustein verpacken, den das
Modell viele Male stapeln kann. Das ist der Transformer-Block.

GPT-2 Small stapelt 12 dieser Blöcke. GPT-3 stapelt 96 davon.
LLaMA 70B stapelt 80. Jedes moderne Sprachmodell entsteht, indem
genau dieser Block immer wieder wiederholt wird. Der Block selbst
ändert sich nicht. Nur die Anzahl ändert sich.

Jeder Block besteht aus zwei Teilen. Zuerst lässt Attention jedes
Wort mit jedem anderen Wort sprechen. Danach lässt ein Feed-Forward-
Netzwerk jedes Wort für sich privat über das Gehörte nachdenken.
Zwischen beiden Teilen normalisieren wir die Eingabe und fügen eine
Skip-Connection hinzu, die das ursprüngliche Signal unverändert
durchlässt.

Die Skip-Connection ist das Geheimnis tiefer Netzwerke. Ohne sie
verschwinden die Gradienten und das Training scheitert schon nach
wenigen Layern. Mit ihr haben die Gradienten eine direkte Autobahn
zurück zur ersten Layer. Deshalb können wir 12 oder 96 oder 100
dieser Blöcke stapeln, und sie alle lernen weiter.

In diesem Notebook bauen wir den vollständigen Transformer-Block
mit RMSNorm zur Normalisierung und SwiGLU als Feed-Forward-Layer.
Das sind die modernen Entscheidungen, die auch LLaMA und Mistral
verwenden. Außerdem übernehmen wir das Attention-Modul, das wir im
letzten Kapitel geschrieben haben, damit dieses Notebook in sich
abgeschlossen bleibt.

## Importe

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Rotary Position Embeddings

Aus Kapitel 4. Wir brauchen das für das Attention-Modul.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = freqs.repeat_interleave(2, dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    @staticmethod
    def rotate_half(x):
        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]
        return torch.stack([-x_odd, x_even], dim=-1).flatten(-2)
    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## Causal Mask

In [ ]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

## Multi-Head Attention

Aus Kapitel 5. Das vollständige Attention-Modul mit RoPE und Causal Masking.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

## RMSNorm

Root-Mean-Square-Normalisierung. Einfacher und schneller als LayerNorm.
Teilt durch den quadratischen Mittelwert (Root Mean Square) des Vektors. Keine Zentrierung nötig.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight

## SwiGLU

Ein Feed-Forward-Netzwerk mit Gating-Mechanismus. Besser als ReLU oder GELU
im großen Maßstab. Ein Pfad erzeugt Werte. Der andere Pfad erzeugt Gates,
die steuern, wie viel von jedem Wert durchgelassen wird.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, expansion_factor=4):
        super().__init__()
        hidden_dim = expansion_factor * d_model
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

## Transformer-Block

Hier wird alles zusammengesetzt. Zwei Sublayer, jeweils mit Pre-Norm
und einer Residual Connection. Attention mischt Informationen zwischen
Tokens. SwiGLU verarbeitet jedes Token unabhängig.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## Den Block testen

Erstelle einen Transformer-Block im Maßstab von GPT-2 und führe ihn aus.

In [ ]:
d_model = 768
num_heads = 12
seq_len = 64
batch_size = 4

block = TransformerBlock(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len, x.device)

output = block(x, mask)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Shapes match: {x.shape == output.shape}")
print()

diff = (output - x).abs().mean().item()
print(f"Mean change after block: {diff:.4f}")
print(f"The block changed the input (it did something)")
print()

params = sum(p.numel() for p in block.parameters())
print(f"Parameters per block: {params:,}")
print(f"For a 12 layer GPT-2 Small: {params * 12:,} params in all blocks")

## Die Residual Connection in Aktion

Setze Attention und FFN so, dass sie null ausgeben. Die Eingabe sollte
dank der Skip-Connection unverändert durchgereicht werden.

In [ ]:
block.eval()
with torch.no_grad():
    for p in block.attention.parameters():
        p.zero_()
    for p in block.ffn.parameters():
        p.zero_()

x = torch.randn(1, 4, d_model)
mask = create_causal_mask(4, x.device)
output = block(x, mask)

diff = (output - x).abs().max().item()
print(f"Max difference when weights are zero: {diff:.10f}")
print(f"Input passes through unchanged (the residual works)")